# Database Content Viewer

This notebook provides a **read-only**, visual overview of all records currently stored in the database. It accesses data safely using the Repositories without making any modifications.

In [ ]:
# Import required modules
from domain.config import DB_URL
from domain.repositories.unit_of_work import UnitOfWorkFactory

from domain.models import Container
from domain.repositories.patient_repository import PatientRepository
from domain.repositories.staff_repository import StaffRepository
from domain.repositories.sample_type_repository import SampleTypeRepository
from domain.repositories.protocol_repository import ProtocolRepository
from domain.repositories.research_project_repository import ResearchProjectRepository
from domain.repositories.sample_repository import SampleRepository
from domain.repositories.quality_control_repository import QualityControlRepository
from domain.repositories.log_temperature_repository import LogTemperatureRepository
from domain.repositories.base_repository import BaseRepository

import warnings
warnings.filterwarnings('ignore')

# Initialize Unit of Work Factory
uow_factory = UnitOfWorkFactory(DB_URL)

# Visual Helpers
def print_header(title):
    print(f"\n{'='*70}")
    print(f"{title.center(70)}")
    print(f"{'='*70}")
    
def print_separator():
    print(f"  {'-'*66}")

## 1. Core Entities

In [ ]:
# Read-only access to Core Entities (Patient, Staff, SampleType, Container, Protocol)
with uow_factory.create() as uow:
    patient_repo = PatientRepository(uow.session)
    staff_repo = StaffRepository(uow.session)
    stype_repo = SampleTypeRepository(uow.session)
    container_repo = BaseRepository[Container, int](uow.session, Container)
    protocol_repo = ProtocolRepository(uow.session)
    
    # --- PATIENTS ---
    print_header("PATIENTS")
    patients = patient_repo.get_all()
    if not patients:
        print("  [Empty] No patients found.")
    for p in patients:
        status = '🟢 Active' if p.active else '🔴 Inactive'
        print(f"  [{p.code}] {p.name} {p.lastname} | {status} | Born: {p.birth_date} | Test: {p.test}")
        
    # --- STAFF ---
    print_header("STAFF MEMBERS")
    staff = staff_repo.get_all()
    if not staff:
        print("  [Empty] No staff found.")
    for s in staff:
        status = '🟢 Active' if s.active else '🔴 Inactive'
        print(f"  [{s.code}] {s.name} {s.lastname} | Role: {s.role.upper()} | {status}")
        
    # --- SAMPLE TYPES & CONTAINERS ---
    print_header("SAMPLE TYPES & CONTAINERS")
    stypes = stype_repo.get_all()
    print("  Sample Types:")
    for st in stypes:
        print(f"   - ID: {st.id} | {st.type_name}")
    
    print_separator()
    print("  Containers:")
    containers = container_repo.get_all()
    for c in containers:
        print(f"   - [{c.code}] {c.type_name}")
        
    # --- PROTOCOLS ---
    print_header("PROTOCOLS")
    protocols = protocol_repo.get_all()
    if not protocols:
        print("  [Empty] No protocols found.")
    for pr in protocols:
        reviewed = f" | Reviewed by ID: {pr.reviewed_by_id}" if pr.reviewed_by_id else ""
        print(f"  [{pr.code}] {pr.name}{reviewed}")
        if pr.description:
            print(f"      Desc: {pr.description[:50]}...")

## 2. Samples and Tracking

In [ ]:
# Read-only access to Samples, Quality Controls, and Temperature Logs
with uow_factory.create() as uow:
    sample_repo = SampleRepository(uow.session)
    qc_repo = QualityControlRepository(uow.session)
    log_repo = LogTemperatureRepository(uow.session)
    
    print_header("SAMPLES OVERVIEW (WITH QC & LOGS)")
    samples = sample_repo.get_all()
    if not samples:
        print("  [Empty] No samples found.")
        
    status_map = {'pending': '🕒', 'in_process': '⚙️', 'analyzed': '✅', 'rejected': '❌', 'archived': '📦'}
    
    for smp in samples:
        icon = status_map.get(smp.status, '🔹')
        print(f"\n  {icon} [{smp.code}] | Status: {smp.status.upper()} | Vol: {smp.volume}ml")
        print(f"      Extracted: {smp.extraction_date} | Patient ID: {smp.id_patient} | Type ID: {smp.id_sample_type} | Container ID: {smp.id_container}")
        
        # Fetch related Protocols
        full_smp = sample_repo.get_full(smp.code)
        if full_smp and full_smp.protocols:
            print(f"      > 📝 Protocols: {len(full_smp.protocols)} linked (e.g. {full_smp.protocols[0].code})")
        
        # Fetch related Quality Control explicitly
        qc = qc_repo.get_by_sample_code(smp.code)
        if qc:
            qc_icon = '🟢' if qc.result == 'approved' else '🔴' if qc.result == 'rejected' else '🟡'
            print(f"      > {qc_icon} Quality Control: {qc.result.upper()} (Purity: {qc.purity}%, Conc: {qc.concentration})")
        else:
            print(f"      > ⚪ Quality Control: Not Performed")

        # Fetch related Temperature Logs explicitly
        logs = log_repo.get_by_sample_code(smp.code)
        if logs:
            print(f"      > 🌡️ Temp Logs ({len(logs)} records): Latest = {logs[-1].temperature}°C on {logs[-1].reading_date}")
        else:
            print(f"      > 🌡️ Temp Logs: None")

## 3. Research Projects

In [ ]:
# Read-only access to Research Projects and associations
with uow_factory.create() as uow:
    project_repo = ResearchProjectRepository(uow.session)
    
    print_header("RESEARCH PROJECTS")
    projects = project_repo.get_all()
    if not projects:
        print("  [Empty] No research projects found.")
        
    for proj in projects:
        # Eager load the team and samples using repository method
        full_proj = project_repo.get_with_team(proj.id)
        print(f"\n  🔬 [{proj.id}] {full_proj.project_name}")
        print(f"      Start Date: {full_proj.start_date}")
        print(f"      Description: {full_proj.description or 'N/A'}")
        
        # Team members list
        if full_proj.staff_members:
            print(f"      Team Members ({len(full_proj.staff_members)}):")
            for member in full_proj.staff_members:
                print(f"         - {member.name} {member.lastname} (Staff Role: {member.role})")
        else:
            print("      Team Members: None")
            
        # Samples list
        if full_proj.samples:
            print(f"      Linked Samples ({len(full_proj.samples)}):")
            for smp in full_proj.samples:
                print(f"         - {smp.code}")
        else:
            print("      Linked Samples: None")
    print("\n")